Setup code - this notebook should be using cuda

In [ ]:
! pip install ultralytics

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Train YOLO model segment

In [ ]:
# prompt: train YOLO11x model on a custom dataset

from ultralytics import YOLO
import os
from itertools import product

# Assuming your dataset is in the standard YOLO format and located at '/content/mydataset'
# The structure should be something like:
# /content/mydataset/
#   train/
#     images/
#     labels/
#   valid/
#     images/
#     labels/
#   data.yaml

# Define the path to your data.yaml file
data_yaml_path = './VisDrone/data.yaml'

#definirani parametri
param_grid = {
    'epochs': [5, 10,20],
    'imgsz': [416, 640, 860],
    'batch': [4, 8, 16]
}



# Check if the data.yaml file exists
if not os.path.exists(data_yaml_path):
    print(f"Error: {data_yaml_path} not found. Please make sure your dataset is correctly placed.")
else:
    

    for epochs, imgsz, batch in product(param_grid['epochs'], param_grid['imgsz'], param_grid['batch']):

        # Load a YOLOv11l from internet
        model = YOLO('yolo11l.pt')  # You can also use 'yolo11m.pt' for a smaller model
        #print(f'epoch: {epochs}, imgsz: {imgsz}, batch: {batch}')

        # Train the model
        # Specify the data file, number of epochs, and image size
        results = model.train(data=data_yaml_path, epochs=epochs, imgsz=imgsz, batch=batch, patience=epochs//2) #416 (also try 640 when enough HW) should capture enough of parking lot details, whilst keepinng enough space for model to run on colab

        # You can access training results and other information from the 'results' object
        # For example, results.save_dir will show where the training results are saved
        print(f"Training results saved to: {model.trainer.save_dir}")


epoch: 5, imgsz: 416, batch: 4
epoch: 5, imgsz: 416, batch: 8
epoch: 5, imgsz: 416, batch: 16
epoch: 5, imgsz: 640, batch: 4
epoch: 5, imgsz: 640, batch: 8
epoch: 5, imgsz: 640, batch: 16
epoch: 5, imgsz: 860, batch: 4
epoch: 5, imgsz: 860, batch: 8
epoch: 5, imgsz: 860, batch: 16
epoch: 10, imgsz: 416, batch: 4
epoch: 10, imgsz: 416, batch: 8
epoch: 10, imgsz: 416, batch: 16
epoch: 10, imgsz: 640, batch: 4
epoch: 10, imgsz: 640, batch: 8
epoch: 10, imgsz: 640, batch: 16
epoch: 10, imgsz: 860, batch: 4
epoch: 10, imgsz: 860, batch: 8
epoch: 10, imgsz: 860, batch: 16
epoch: 15, imgsz: 416, batch: 4
epoch: 15, imgsz: 416, batch: 8
epoch: 15, imgsz: 416, batch: 16
epoch: 15, imgsz: 640, batch: 4
epoch: 15, imgsz: 640, batch: 8
epoch: 15, imgsz: 640, batch: 16
epoch: 15, imgsz: 860, batch: 4
epoch: 15, imgsz: 860, batch: 8
epoch: 15, imgsz: 860, batch: 16
epoch: 20, imgsz: 416, batch: 4
epoch: 20, imgsz: 416, batch: 8
epoch: 20, imgsz: 416, batch: 16
epoch: 20, imgsz: 640, batch: 4
epoch: 

In [3]:
# Access the path to the best weights after training
best_weights_path = model.trainer.best


# Copy the best weights file to your custom path
import shutil
# Define the path on Google Drive where you want to save the weights
drive_save_path = './yolo11m_parkman_weights.pt' # Replace with your desired path

# Copy the best weights file to Google Drive
shutil.copy(best_weights_path, drive_save_path)

print(f"Best weights saved to disk at: {drive_save_path}")



# Example of how to load the custom saved weights in different code:
# from ultralytics import YOLO
#
# # Define the path to your custom saved weights
# custom_loaded_weights_path = '/content/my_best_yolo11x_weights.pt'
#
# # Load the model with the custom weights
# loaded_model = YOLO(custom_loaded_weights_path)
#
# print("Model loaded successfully from custom weights.")
# # You can now use the loaded_model for inference or further training

Best weights saved to disk at: ./yolo11m_parkman_weights.pt


# Test YOLO on unseen image

In [ ]:
from ultralytics import YOLO
import os
import cv2
#test on example image

image_path="./park4.jpg"

model=YOLO("./yolo11m_parkman_weights.pt")


results=model.predict(image_path)

cv2.imwrite("Yolo_detects.jpg",results[0].plot())


image 1/1 c:\Users\BenjaminJ\Desktop\Faks\Diplomski\APVO\Parkman\YOLOTrain\park4.jpg: 480x640 5 space-occupieds, 29.6ms
Speed: 2.0ms preprocess, 29.6ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)


True